In [ ]:
from platform import python_version
print(python_version())

In [ ]:
!echo $CONDA_DEFAULT_ENV

## Pathways' Discover

In [1]:
import os, sys, pickle

import numpy as np
import pandas as pd
pd.set_option('display.width', 100)
pd.set_option('max_colwidth', 80)
import yaml

import seaborn as sns
sns.set_context("notebook", font_scale=1.4)

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
%matplotlib inline

sys.path.insert(1, '../src/')

from Basic import *
from MTD_lib import *
from gemini_lib import *
# from enricher_lib import *
# from config_lib import *

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.precision", 3)
from IPython.display import display, HTML
# display(HTML("<style>.container { width:100% !important; }</style>"))
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

# !pip3 install pyyaml
with open('params.yml', 'r') as file:
    dic_yml = yaml.safe_load(file)

In [2]:
root_chibe = dic_yml['root_chibe']
root_colab = dic_yml['root_colab']
root0 = dic_yml['root0']

from dotenv import load_dotenv
load_dotenv()

email = os.getenv('email')

try:
    i_project=dic_yml['i_project']
except:
    i_project=0

print(">>> i_project", i_project)

project_list = dic_yml['project_list']
n = len(project_list)
project = project_list[i_project]

s_project_list = dic_yml['s_project_list']
s_project = s_project_list[i_project]
assert n==len(project_list), f"Error project_list: there are {n} projects"

disease = dic_yml['disease']
gene_protein = dic_yml['gene_protein']
s_omics = dic_yml['s_omics']

has_age = dic_yml['has_age']
has_gender = dic_yml['has_gender']

exp_normalization = dic_yml['exp_normalization']
normalization = 'quantile_norm' if exp_normalization == True else 'not_normalized'

abs_lfc_cutoff_inf = dic_yml['abs_lfc_cutoff_inf']
s_pathw_enrichm_method = dic_yml['s_pathw_enrichm_method']
num_min_degs_for_ptw_enr = dic_yml['num_min_degs_for_ptw_enr']

tolerance_pPMI = dic_yml['tolerance_pPMI']
type_sat_ptw_index = dic_yml['type_sat_ptw_index']
saturation_lfc_param = dic_yml['saturation_lfc_param']

pval_pathway_cutoff = dic_yml['pval_pathway_cutoff']
fdr_pathway_cutoff = dic_yml['fdr_pathway_cutoff']
num_of_genes_cutoff = dic_yml['num_of_genes_cutoff']
enr_db_list = dic_yml['enr_db_list']

case_list = dic_yml['case_list']
dic_case_list = dic_yml['dic_case_list']

std_filename      = dic_yml['std_filename']
std_filename_list = dic_yml['std_filename_list']

min_lfc_modulation = dic_yml['min_lfc_modulation']
num_of_genes_list  = dic_yml['num_of_genes_list']
pPMI_normalized  = dic_yml['pPMI_normalized']

#--- max len for formatting purposes
s_len_case  = dic_yml['s_len_case']

n_sentences = dic_yml['n_sentences']
run_list = dic_yml['run_list']
chosen_model_list = dic_yml['chosen_model_list']
i_dfp_list = dic_yml['i_dfp_list']
chosen_model_sampling = dic_yml['chosen_model_sampling']

fdr_ptw_cutoff_list = np.arange(0.05, 0.80, 0.05)
lfc_list = np.round(np.arange(1.0, -0.01, -.025), 3)
fdr_list = np.arange(0.05, 0.76, .01)

cfg = Config(project, s_project, case_list, root0)
case = case_list[0]

n_genes_annot_ptw, n_degs, n_degs_in_ptw, n_degs_not_in_ptw, degs_in_all_ratio = -1,-1,-1,-1,-1
abs_lfc_cutoff, fdr_lfc_cutoff, n_degs, n_degs_up, n_degs_dw = cfg.get_best_lfc_cutoff(case, 'not_normalized')

print(f"project '{project}', s_project '{s_project}'")
print(f"G/P LFC cutoffs: lfc={abs_lfc_cutoff:.3f}; fdr={fdr_lfc_cutoff:.3f}")
print(f"Pathway cutoffs: pval={pval_pathway_cutoff:.3f}; fdr={fdr_pathway_cutoff:.3f}; num of genes={num_of_genes_cutoff}")

>>> i_project 0
project 'Medulloblastoma microarray study', s_project 'medulloblastoma'
G/P LFC cutoffs: lfc=1.000; fdr=0.050
Pathway cutoffs: pval=0.050; fdr=0.050; num of genes=3


In [3]:
mtd = MTD(disease=disease, gene_protein=gene_protein, s_omics=s_omics, project=project, s_project=s_project, root0=root0,
          case_list=case_list, dic_case_list=dic_case_list, has_age=has_age, has_gender=has_gender, exp_normalization=exp_normalization,
          std_filename=std_filename, std_filename_list=std_filename_list,
          geneset_num=0, num_min_degs_for_ptw_enr=num_min_degs_for_ptw_enr,
          tolerance_pPMI=tolerance_pPMI, s_pathw_enrichm_method=s_pathw_enrichm_method,
          root_colab=root_colab, root_chibe=root_chibe, 
          abs_lfc_cutoff_inf=abs_lfc_cutoff_inf, fdr_ptw_cutoff_list=fdr_ptw_cutoff_list,
          num_of_genes_list=num_of_genes_list, lfc_list=lfc_list, fdr_list=fdr_list, 
          min_lfc_modulation=min_lfc_modulation, type_sat_ptw_index=type_sat_ptw_index,
          saturation_lfc_param=saturation_lfc_param, enr_db_list=enr_db_list, pPMI_normalized=pPMI_normalized)

case = case_list[0]
print(">>>", case)

mtd.cfg.set_default_best_lfc_cutoff(mtd.normalization, abs_lfc_cutoff=1, fdr_lfc_cutoff=0.05)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, prompt_verbose=True, verbose=False)
print("\nEcho Parameters:")
mtd.echo_parameters()

Start opening tables ....
Building synonym dictionary ...

$$$ Removed: ./figures/
>>> WNT
>>> case WNT
	DEGs 3448
		Up (#1441)
		Dw (#2007)

Up-regulated per biotype
                               biotype    n
0                                  LNC  321
1                                  TEC    1
2                                  UNK   25
3                            hasSymbol  128
4                               lncRNA   19
5                                ncRNA   47
6                 processed_pseudogene    2
7                       protein_coding  882
8                               scaRNA    2
9                               snoRNA    9
10      transcribed_unitary_pseudogene    1
11  transcribed_unprocessed_pseudogene    2
12              unprocessed_pseudogene    2

Down-regulated per biotype
                               biotype     n
0                                  LNC   223
1                                  UNK    23
2                            hasSymbol   217
3        

In [4]:
mtd.abs_lfc_cutoff, mtd.fdr_lfc_cutoff, mtd.min_lfc_modulation

(np.float64(0.825), np.float64(0.33), 0.4)

In [5]:
len(mtd.dflfc)

3448

In [6]:
mtd.df_enr.head(6)

,pathway,pathway_id,pval,fdr,odds_ratio,combined_score,genes,num_of_genes
0,Neuronal System,R-HSA-112316,3.926e-20,7.047e-17,3.023,135.087,"['RAB3A', 'ABAT', 'RIMS1', 'HOMER3', 'PRKACB', 'PPFIA2', 'PRKCG', 'NSF', 'KC...",119
1,Extracellular Matrix Organization,R-HSA-1474244,3.066e-13,2.752e-10,2.806,80.858,"['DDR1', 'APP', 'COL18A1', 'PTPRS', 'ITGB4', 'ELN', 'COL12A1', 'ICAM2', 'TNC...",83
2,Signal Transduction,R-HSA-162582,1.912e-11,1.144e-08,1.487,36.695,"['IFT172', 'SPINT2', 'FZD10', 'MYLK', 'GJA1', 'PSMD6', 'SOX17', 'MYC', 'EPST...",427
3,Protein-protein Interactions at Synapses,R-HSA-6794362,4.919e-09,2.207e-06,4.142,79.238,"['GRIA1', 'NLGN1', 'PTPRS', 'NRXN1', 'NRXN3', 'LIN7A', 'GRM1', 'EPB41L5', 'G...",32
4,Transmission Across Chemical Synapses,R-HSA-112315,7.038e-09,2.527e-06,2.419,45.414,"['RAB3A', 'RASGRF1', 'GRIK3', 'PRKAG2', 'ABAT', 'KIF17', 'SLC6A4', 'RIMS1', ...",67
5,Potassium Channels,R-HSA-1296071,2.800e-08,8.376e-06,3.572,62.122,"['HCN4', 'KCNK9', 'KCNC2', 'KCNA1', 'KCNC4', 'KCNA4', 'KCNA5', 'KCNV1', 'GNG...",34


### Summary in text

In [7]:
mtd.geneset_num

0

In [8]:
mtd.set_db(mtd.geneset_num, verbose=True)

>>> Reactome_Pathways_2024


### Cases

In [9]:
mtd.case_list

['WNT', 'G4']

In [10]:
i = 0
case = case_list[i]
print(">>>", case)
ret, degs, degs_ensembl, dfdegs = mtd.open_case(case, verbose=False)

df_enr = mtd.df_enr
print(len(df_enr))
df_enr.head(3)

>>> WNT
104


,pathway,pathway_id,pval,fdr,odds_ratio,combined_score,genes,num_of_genes
0,Neuronal System,R-HSA-112316,3.926e-20,7.047e-17,3.023,135.087,"['RAB3A', 'ABAT', 'RIMS1', 'HOMER3', 'PRKACB', 'PPFIA2', 'PRKCG', 'NSF', 'KC...",119
1,Extracellular Matrix Organization,R-HSA-1474244,3.066e-13,2.752e-10,2.806,80.858,"['DDR1', 'APP', 'COL18A1', 'PTPRS', 'ITGB4', 'ELN', 'COL12A1', 'ICAM2', 'TNC...",83
2,Signal Transduction,R-HSA-162582,1.912e-11,1.144e-08,1.487,36.695,"['IFT172', 'SPINT2', 'FZD10', 'MYLK', 'GJA1', 'PSMD6', 'SOX17', 'MYC', 'EPST...",427


### DEGs to text

In [11]:
force=True
verbose=False

text = mtd.degs_to_text_all_cases_summary(force=force, verbose=verbose)
print(text)

# Project Medulloblastoma microarray study: microarray for Medulloblastoma
# Calculated in 14/11/2025 18:50:23
# geneset 0 - Reactome_Pathways_2024

0) WNT ('WNT') 

BCA's LFC DEG cutoffs: abs(lfc)=0.825; fdr=0.330

	There are a total of 3448 DEGs, and 2446 have ensembl_id

		All DEGs Up (1441): A1BG-AS1, AACS, AADAT, AANAT, ABCA1, ABCC1, ABHD12B, ABHD14B, ACAA2, ACAD11, ACCS, ACOX2, ACP6, ACSS2, ACTC1, ACTL6A, ACTR3B, ACTR5, ADAM12, ADAM19, ADAMTS4, ADAMTSL1, ADAT3, ADCY3, ADPRH, AEN, AGXT2L2, AHCY, AHSA2, AKR7L, ALDH16A1, ALDH3B1, ALDH4A1, ALG1L2, ALG3, ALK, ALPL, AMHR2, ANGEL1, ANKDD1A, ANKHD1-EIF4EBP3, ANKMY1, ANKRD16, ANO2, ANTXR2, APCDD1, APLNR, APP, ARHGEF19, ASB13, ASXL2, ATAD3C, ATOH8, ATP13A1, ATP1B3, ATP2B1, ATP6V0D1, ATP6V1C2, ATP6V1E2, ATXN2L, AXIN2, B3GALNT2, B3GNT2, B3GNT8, B3GNT9, BAG2, BAMBI, BAZ1A, BHMT2, BLID, BMF, BMI1, BMP4, BOP1, BORA, BRCA1, BRIP1, BSG, C10orf114, C10orf2, C11orf49, C11orf82, C12orf33, C12orf5, C13orf33, C14orf101, C14orf142, C14orf48, C15orf54, 

In [12]:
fname, fname_cutoff = mtd.set_enrichment_name()
filefull1 = os.path.join(mtd.root_enrich, fname)
filefull2 = os.path.join(mtd.root_enrich, fname_cutoff)

print(os.path.exists(filefull1), filefull1)
print(os.path.exists(filefull2), filefull2)

True ../../colaboracoes/medulloblastoma/projects/medulloblastoma/enrichment_analyses/enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_G4_x_ctrl_not_normalized_cutoff_lfc_1.000_fdr_0.160.tsv
True ../../colaboracoes/medulloblastoma/projects/medulloblastoma/enrichment_analyses/enricher_Reactome_Pathways_2024_medulloblastoma_microarray_for_G4_x_ctrl_not_normalized_cutoff_lfc_1.000_fdr_0.160_pathway_pval_0.050_fdr_0.750_num_genes_3.tsv


### Most positive and most negative "pseudo-Pathways Modulation Index" (pPMI)

  - method: calc_all_pPMI() / calc_pathway_gene_index_per_case()
    
    - _type 'discrete'
      - mod_up = #mod_up_in_pathway if #mod_up_in_pathway > 0 else 1/2
      - mod_dw = #mod_dw_in_pathway if #mod_dw_in_pathway > 0 else 1/2
    
    - _type 'linear'
      - mod_up = 0.001 if no_data else sum of lfc
      - mod_dw = 0.001 if no_data else sum of lfc
  
    - _type 'linear_sat' (default saturation = 3)
      - mod_up = sum(lfc_positive (if lfc > +saturation = +saturation ))
      - mod_dw = sum(lfc_negative (if lfc < -saturation = -saturation ))
  
  - pPMI

    - ratio = np.abs(mod_up / mod_dw)  
    - pathway_mod_gene_index = log2(ratio)

In [13]:
lista = [-4., -3, -2, -1.5, -1, -0.5]
saturation = 3
[saturation if x < -saturation else x for x in lista]

[3, -3, -2, -1.5, -1, -0.5]

In [14]:
mtd.type_sat_ptw_index

'linear_sat'

#### calc_pPMI_summary

In [15]:
verbose=False
force=False

dff, text_all, df_sum_ptw = mtd.calc_pPMI_summary(force=force, verbose=verbose)
print(len(dff))

lines = [0, 5, 10]
dff.iloc[lines].T

444


,0,5,10
case,G4,G4,G4
pathway_id,R-HSA-1971475,R-HSA-114452,R-HSA-445717
pathway,A Tetrasaccharide Linker Sequence Is Required for GAG Synthesis,Activation of BH3-only Proteins,Aquaporin-mediated Transport
pPMI_total,-1.921,-0.115,-1.282
ratio_up_dw,0.264,0.923,0.411
pPMI_norm,-0.751,-0.285,-0.841
ratio_norm_up_dw,0.594,0.821,0.558
lfc_pPMI_up,2.876,9.986,11.779
lfc_pPMI_dw,-10.886,-10.818,-28.634
n_genes_annot_in_pathway,26,30,52


In [16]:
dff.columns

Index(['case', 'pathway_id', 'pathway', 'pPMI_total', 'ratio_up_dw', 'pPMI_norm',
       'ratio_norm_up_dw', 'lfc_pPMI_up', 'lfc_pPMI_dw', 'n_genes_annot_in_pathway',
       'n_mod_in_pathway', 'perc_mod_in_pathway', 'n_mod_up_in_pathway', 'n_mod_dw_in_pathway',
       'fdr', 'pval', 'mod_up_in_pathway', 'mod_dw_in_pathway', 'lfc_up', 'lfc_dw',
       'genes_annot_in_pathway'],
      dtype='object')

In [17]:
i=0
case = case_list[i]
print(">>>", case)

cols = ['case', 'pathway_id', 'pathway', 'pPMI_total', 'ratio_up_dw', 'pPMI_norm',
       'ratio_norm_up_dw', 'lfc_pPMI_up', 'lfc_pPMI_dw', 'n_genes_annot_in_pathway',
       'n_mod_in_pathway', 'n_mod_up_in_pathway', 'n_mod_dw_in_pathway',
       'fdr', 'mod_up_in_pathway', 'mod_dw_in_pathway', 'lfc_up', 'lfc_dw']

df2 = dff[ (dff.case==case) & (~pd.isnull(dff.fdr)) ]
print(len(df2))
df2[cols]

>>> WNT
221


,case,pathway_id,pathway,pPMI_total,ratio_up_dw,pPMI_norm,ratio_norm_up_dw,lfc_pPMI_up,lfc_pPMI_dw,n_genes_annot_in_pathway,n_mod_in_pathway,n_mod_up_in_pathway,n_mod_dw_in_pathway,fdr,mod_up_in_pathway,mod_dw_in_pathway,lfc_up,lfc_dw
1,WNT,R-HSA-1971475,A Tetrasaccharide Linker Sequence Is Required for GAG Synthesis,-0.891,0.539,-0.891,0.539,7.499,-13.905,26,16,8,8,7.047e-17,"['XYLT2', 'GPC1', 'CSPG4', 'GPC2', 'CSPG5', 'GPC6', 'SDC2', 'HSPG2']","['SDC4', 'VCAN', 'B3GAT2', 'B3GAT1', 'AGRN', 'DCN', 'SDC3', 'BCAN']","[0.844, 0.519, 0.84, 1.032, 1.067, 1.42, 1.089, 0.688]","[-1.471, -1.846, -2.676, -1.558, -1.107, -1.237, -2.4, -1.61]"
2,WNT,R-HSA-2033519,Activated Point Mutants of FGFR2,-2.099,0.233,-0.514,0.700,1.855,-7.947,17,8,2,6,7.047e-17,"['FGF8', 'FGF22']","['FGF5', 'FGF1', 'FGFR2', 'FGF4', 'FGF9', 'FGF3']","[1.169, 0.685]","[-2.829, -1.835, -0.6, -0.403, -1.503, -0.777]"
4,WNT,R-HSA-114452,Activation of BH3-only Proteins,0.321,1.249,0.321,1.249,14.207,-11.377,30,20,10,10,7.047e-17,"['TP53BP2', 'AKT2', 'BMF', 'E2F1', 'AKT3', 'BCL2L11', 'TP53', 'SFN', 'BAD', ...","['YWHAH', 'PPP3CC', 'BCL2', 'BBC3', 'DYNLL1', 'YWHAG', 'YWHAE', 'TP73', 'MAP...","[0.835, 0.792, 2.674, 1.98, 0.779, 0.611, 0.939, 3.672, 0.762, 1.161]","[-2.344, -2.496, -1.191, -1.159, -0.422, -0.924, -0.736, -0.825, -0.594, -0...."
7,WNT,R-HSA-442755,Activation of NMDA Receptors and Postsynaptic Events,-3.205,0.108,-0.972,0.510,7.221,-66.593,93,57,10,47,7.047e-17,"['GIT1', 'CAMK1', 'PRKAB1', 'PRKAR2A', 'DLG1', 'GRIN2C', 'PPM1F', 'TUBAL3', ...","['TUBB4A', 'NRGN', 'TUBA1B', 'GRIN2B', 'TUBA4A', 'LIN7B', 'ADCY8', 'CAMK4', ...","[0.73, 0.471, 0.472, 0.444, 0.727, 0.465, 1.373, 0.966, 1.117, 0.457]","[-1.723, -0.772, -0.514, -1.756, -1.382, -0.786, -1.107, -1.976, -0.543, -3...."
9,WNT,R-HSA-187015,Activation of TRKA Receptors,-0.580,0.669,-0.580,0.669,2.126,-3.178,6,4,2,2,7.047e-17,"['NGF', 'NTRK1']","['NTRK2', 'ADCYAP1']","[0.411, 1.714]","[-1.827, -1.35]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
434,WNT,R-HSA-8979227,Triglyceride Metabolism,-1.224,0.428,-0.738,0.599,7.256,-16.946,38,24,10,14,7.047e-17,"['AGMO', 'PPP1CC', 'PPP1CA', 'DGAT2', 'FABP4', 'GPAM', 'LPIN1', 'DGAT1', 'GP...","['FABP7', 'MOGAT1', 'PLIN1', 'MOGAT3', 'PRKACB', 'FABP6', 'PRKACG', 'FABP1',...","[0.495, 0.526, 0.515, 1.158, 0.428, 0.48, 1.661, 0.428, 0.572, 0.993]","[-1.467, -0.497, -0.403, -0.456, -1.827, -2.915, -0.825, -0.647, -1.405, -0...."
437,WNT,R-HSA-5218921,VEGFR2 Mediated Cell Proliferation,-2.745,0.149,-1.008,0.497,2.334,-15.649,19,13,3,10,7.047e-17,"['PRKCD', 'PLCG1', 'SPHK1']","['ITPR3', 'AHCYL1', 'PRKCZ', 'ITPR1', 'CALM1', 'KDR', 'PRKCA', 'ITPR2', 'PRK...","[0.912, 0.968, 0.455]","[-0.532, -1.042, -0.845, -2.165, -0.722, -1.318, -1.081, -2.023, -5.833, -0.92]"
438,WNT,R-HSA-432040,Vasopressin Regulates Renal Water Homeostasis via Aquaporins,-0.438,0.738,-0.531,0.692,13.293,-18.007,43,31,16,15,7.047e-17,"['AVP', 'GNGT1', 'RAB11FIP2', 'ADCY5', 'GNG3', 'PRKAR2A', 'AQP1', 'ADCY6', '...","['GNG11', 'GNG5', 'GNG7', 'ADCY8', 'PRKACB', 'ADCY4', 'PRKACG', 'AQP4', 'PRK...","[0.41, 0.555, 2.134, 0.987, 0.561, 0.444, 0.619, 0.601, 1.222, 1.471, 0.551,...","[-0.939, -0.419, -1.497, -1.107, -1.827, -0.634, -0.825, -2.766, -0.638, -1...."
440,WNT,R-HSA-1296072,Voltage Gated Potassium Channels,-1.158,0.448,-0.088,0.941,15.566,-34.747,43,31,10,21,7.047e-17,"['KCNH4', 'KCNQ5', 'KCNH6', 'KCNH8', 'KCNH2', 'KCNA5', 'KCNQ1', 'KCNG3', 'KC...","['KCNC3', 'KCNS3', 'KCNC4', 'KCNQ3', 'KCND2', 'KCNS1', 'KCNH7', 'KCND3', 'KC...","[0.935, 0.682, 2.873, 2.111, 0.918, 2.505, 0.618, 1.268, 2.863, 0.795]","[-0.983, -2.223, -2.033, -0.616, -5.355, -1.848, -1.161, -1.376, -0.628, -2...."


In [18]:
df2[cols].head(4).T

,1,2,4,7
case,WNT,WNT,WNT,WNT
pathway_id,R-HSA-1971475,R-HSA-2033519,R-HSA-114452,R-HSA-442755
pathway,A Tetrasaccharide Linker Sequence Is Required for GAG Synthesis,Activated Point Mutants of FGFR2,Activation of BH3-only Proteins,Activation of NMDA Receptors and Postsynaptic Events
pPMI_total,-0.891,-2.099,0.321,-3.205
ratio_up_dw,0.539,0.233,1.249,0.108
pPMI_norm,-0.891,-0.514,0.321,-0.972
ratio_norm_up_dw,0.539,0.7,1.249,0.51
lfc_pPMI_up,7.499,1.855,14.207,7.221
lfc_pPMI_dw,-13.905,-7.947,-11.377,-66.593
n_genes_annot_in_pathway,26,17,30,93


In [19]:
df2.tail(4).T

,437,438,440,443
case,WNT,WNT,WNT,WNT
pathway_id,R-HSA-5218921,R-HSA-432040,R-HSA-1296072,R-HSA-5140745
pathway,VEGFR2 Mediated Cell Proliferation,Vasopressin Regulates Renal Water Homeostasis via Aquaporins,Voltage Gated Potassium Channels,"WNT5A-dependent Internalization of FZD2, FZD5 and ROR2"
pPMI_total,-2.745,-0.438,-1.158,0.006
ratio_up_dw,0.149,0.738,0.448,1.004
pPMI_norm,-1.008,-0.531,-0.088,1.006
ratio_norm_up_dw,0.497,0.692,0.941,2.009
lfc_pPMI_up,2.334,13.293,15.566,6.71
lfc_pPMI_dw,-15.649,-18.007,-34.747,-6.681
n_genes_annot_in_pathway,19,43,43,13


In [20]:
df2.iloc[-1].genes_annot_in_pathway

"['FZD2', 'FZD5', 'WNT5A', 'AP2B1', 'CLTC', 'CLTB', 'AP2A2', 'CLTA', 'AP2A1', 'AP2S1', 'ROR2', 'ROR1', 'AP2M1']"

### Heatmap by gender
  - plot_pathway_heatmap_by_gender()
    - calc_pivot_pPMI()
      - calc_one_pathway_all_cases_gene_modulations()

### pivot summary
  - calc_pPMI_summary_and_pivot_tables
    - calc_pPMI_summary()
      - calc_all_pPMI():
        - _type == 'discrete': s_saturation = 'none'
        - _type == 'linear':   s_saturation = 'none'
        - _type == 'linear_sat': s_saturation = f'{saturation:.3f}'


In [21]:
dfpiv, dff, text_all, df_sum_ptw = mtd.calc_pPMI_summary_and_pivot_tables(force=False, verbose=False)
print(len(df_sum_ptw))
df_sum_ptw.head(2)

dfpiv

444


case,G4,WNT
pathway,,
A Tetrasaccharide Linker Sequence Is Required for GAG Synthesis,-1.921,-0.891
Activated Point Mutants of FGFR2,-3.535,-2.099
Activation of BH3-only Proteins,-0.115,0.321
Activation of NMDA Receptors and Postsynaptic Events,-1.462,-3.205
Activation of TRKA Receptors,-4.517,-0.580
...,...,...
Triglyceride Metabolism,-0.496,-1.224
VEGFR2 Mediated Cell Proliferation,-2.239,-2.745
Vasopressin Regulates Renal Water Homeostasis via Aquaporins,-1.742,-0.438


In [22]:
want_print = False
sel_pathway_list = []

for pathway in dfpiv.index:
    sel_pathway_list.append(pathway)
    if want_print: print(pathway)

In [23]:
"; ".join(sel_pathway_list)

'A Tetrasaccharide Linker Sequence Is Required for GAG Synthesis; Activated Point Mutants of FGFR2; Activation of BH3-only Proteins; Activation of NMDA Receptors and Postsynaptic Events; Activation of TRKA Receptors; Aquaporin-mediated Transport; Arachidonate Metabolism; Assembly of Collagen Fibrils and Other Multimeric Structures; Basigin Interactions; Beta-catenin Independent WNT Signaling; Binding of TCF LEF CTNNB1 to Target Gene Promoters; CDC42 GTPase Cycle; CLEC7A (Dectin-1) Induces NFAT Activation; Ca-dependent Events; Ca2+ Pathway; CaM Pathway; Calmodulin Induced Events; Cardiac Conduction; Cell Death Signalling via NRAGE, NRIF and NADE; Chondroitin Sulfate Dermatan Sulfate Metabolism; Chylomicron Remodeling; Class B 2 (Secretin Family Receptors); Collagen Biosynthesis and Modifying Enzymes; Collagen Chain Trimerization; Collagen Degradation; Collagen Formation; Constitutive Signaling by Aberrant PI3K in Cancer; DAG and IP3 Signaling; DARPP-32 Events; DNA Replication Initiation

### Heatmap plots prepraration

### Pathways' discover code

In [ ]:
term = 'igf'
term = 'Phagocytosis'
term = 'translational'
term = 'Scaven'
term = 'fibrin'
term = 'Activat'; not_term = 'FGFR'
term = 'FGFR'; not_term = ''
term = 'Biosynthesis'; not_term = ''
term = 'nuclear'; not_term = ''
term = 'Degradation'; not_term = ''
term = 'neur'; not_term = ''
term = 'synap'; not_term = ''
term = 'channel'; not_term = ''
term = 'tabol'; not_term = 'lipid'
term = 'transp'; not_term = 'lipid'
term = 'Interac'; not_term = ''
term = 'collag'; not_term = ''
term = 'GTPase'; not_term = ''
term = 'transduction'; not_term = ''

print(">>>", term)

if isinstance(not_term, str) and not_term != '':
    lista = [x for x in mtd.selected_pivot_pathway_list if term.lower() in x.lower() and not_term.lower() not in x.lower()]
else:
    lista = [x for x in mtd.selected_pivot_pathway_list if term.lower() in x.lower()]

lista = [str(x) for x in np.unique(lista)]
lista

In [24]:
terms = ['igf', 'insulin', 'IGFBP']
terms = ['neutrophil', 'vascular', 'nets', 'degranulation']
terms = ['MHC', 'presentation', 'processing', 'TAP', 'endoplasmic']
terms = ['Vascular', 'Degranulation']

terms = ['Degradation', 'Biosynthesis']
terms = ['synap', 'transm', 'channel']
terms = ['neur', 'gaba', 'colin', 'transm']
terms = ['neur']
terms = ['Secretion', 'insulin', 'panc']
terms = ['adherence', 'junctions']
terms = ['syndrome', 'disease']
terms = ['receptor']
terms = ['translational', 'modification', 'carbox', 'Phosphor', 'phosphat', 'dephosphor', 'glycoz', 'cleavag']
terms = ['catenin', 'wnt', 'CDC42', 'LRP6', 'cadherin', 'DVS', 'DLH', 'LRP', 'GSK', 'AXIN', 'FZD']
terms = ['card', 'heart', 'muscle']
terms = ['Transduction', 'VISION', 'AUDITION', 'sensor']
terms = ['hemostasis', 'neutrophil', 'platelet', 'Aggregation', 'degranulation', 'amyloid', 'fibrin', 'Heparan']
terms = ['ECM', 'matrix', 'Elastic',  'Extracellular Matrix', 'heparan']
terms = ['adherence', 'junction', 'notch']
terms = ['lipid', 'HDL', 'LDL', 'trigl', 'choles']
terms = ['Myelin', 'Schwann', 'neuron']
terms = ['Laminin', 'Integrin', 'adhesion', 'migration']
terms = ['gap', 'junction']
terms = ['complement', 'c3', 'c4', 'c5']
terms = ['apoptosis', 'bh', 'bcl']
terms = ['acqua', 'transport', 'carrier', 'solute']
terms = ['gpcr', 'seven', 'G-protein', 'P2Y', 'FGFR']
terms = ['cytok', 'inflam', 'dectin', 'NFAT', 'NLRP3', 'TNF', 'OSM']
terms = ['apopto', 'inflammasome', 'dectin', 'NFAT', 'NLRP3', 'TNF', 'OSM', 'BH', 'CHEMIO', 'CYTO']
terms = ['scav', 'neurodegen', 'LDL', 'lipoprotein']
terms = ['nuclear', 'hdl', 'hormone', 'cholesterol', 'LXR', 'ABC', 'CETP', 'Apolipoprotein', 'PPAR', 'FXR', 'HRE', 'DNA']
terms = ['DAG', 'IP3', 'PIP2', 'PI-3K', 'PI3K']
terms = ['tyr', 'kinase', 'phospha']
terms = ['Neurotransmitter', 'GABA', 'Glut', 'Dopa', 'Acetyl', 'Purinergic']
terms = ['membrane', 'plasma']

lista=[]
for term in terms:
    lista += [x for x in mtd.selected_pivot_pathway_list if term.lower() in x.lower()]

lista = [str(x) for x in np.unique(lista)]
lista

['Non-integrin membrane-ECM Interactions',
 'Phase 4 - Resting Membrane Potential',
 'Plasma Lipoprotein Assembly, Remodeling, and Clearance',
 'SLC-mediated Transmembrane Transport']

In [ ]:
terms = ['scaveng', 'nuclear']

lista=[]
for term in terms:
    lista += [x for x in mtd.selected_pivot_pathway_list if term.lower() in x.lower()]

lista

In [ ]:
[x for x in set_hemostasis if x not in sel_pathway_list]

In [ ]:
mtd.group_female_list, mtd.group_male_list

In [ ]:
mtd.saturation_lfc_index

### Heatmap plots

In [ ]:
mtd.pPMI_normalized

In [ ]:
set_name = set_hemostasis
set_title = 'Hemostasis'
width=1400
height=1300

if mtd.pPMI_normalized:
    fig, dfpiv_female, dfpiv_male, text_all = mtd.plot_pathway_heatmap_by_gender(set_name, set_title, height=height, width=width, zlim=5, maxLen=75, with_pPMI_obs=False, do_case_translate=True)
else:
    fig, dfpiv_female, dfpiv_male, text_all = mtd.plot_pathway_heatmap_by_gender(set_name, set_title, height=height, width=width, zlim=6, maxLen=75, with_pPMI_obs=False, do_case_translate=True)

print('female', len(dfpiv_female))
print('male', len(dfpiv_male))
if fig: fig.show()

In [ ]:
print(text_all)

In [ ]:
dff = mtd.dff

case = 'g2a_male'
pathway = 'Signaling by BRAF and RAF1 Fusions'

row_dff = dff[ (dff.case == case) & (dff.pathway == pathway)]
row_dff

In [ ]:
dfpiv_female

In [ ]:
for i in range(len(dfpiv_female)):
    row = dfpiv_female.iloc[i]
    pathway = row.name
    for col in row.index:
        val = row[col]
        print(i, pathway, val)
        break

In [ ]:
dfpiv, dff, text_all, df_sum_ptw = mtd.calc_pPMI_summary_and_pivot_tables(verbose=verbose)

In [ ]:
case = 'g2a_male'
pathway = 'Signaling by BRAF and RAF1 Fusions'

row_dff = dff[ (dff.case == case) & (dff.pathway == pathway)]
row = row_dff.iloc[0]

print(row_dff['mod_up_in_pathway'], row_dff['mod_dw_in_pathway'])
row_dff.iloc[0]['mod_up_in_pathway']